# Laboratorio de Aprendizaje Automático - Parte 3

## Inteligencia Artificial - DCIC, UNS (2024)

¡Bienvenidos! En este laboratorio aprenderán conceptos y herramientas para el desarrollo de un proyecto con técnicas de aprendizaje automático. Vamos a ver algunas nociones para el preprocesamiento de datos, junto con la construcción, entrenamiento y generación de predicciones utilizando dos métodos de aprendizaje automático: Árboles de Decisión y Redes Neuronales Artificiales (RNA).

**Recuerde hacer una copia de este notebook en su drive**.

## Plan de Trabajo

En esta tercera parte, nos vamos a enfocar en el desarrollo de modelos usando Redes Neuronales Artificiales (RNAs).

En particular, vamos a construir y entrenar RNAs para abordar un problema de clasificación y usarlo para generar nuevas predicciones.

## Datasets

En el laboratorio vamos a utilizar tres datasets, que pueden **descargar ejecutando la primera celda de código** o desde el siguiente enlace: [datasets](https://drive.google.com/file/d/16AB3ixX9q2S7fHAbiE7id7zj1fCgPbo4/view).

*   Titanic dataset (titanic.csv)
*   Dataset precio celulares (cellphones.csv)
*   Dataset géneros musicales (music.csv)

## Librerías

A lo largo del laboratorio se usan principalmente tres librerías:
*   **Pandas**: Permite utilizar y analizar los datos. Link: https://pandas.pydata.org/docs/
*   **Scikit-learn**: Permite manipular los datos e implementar diversos algoritmos de *Machine Learning*. Link: https://scikit-learn.org/stable/
*   **Keras**: Permite implementar Redes Neuronales Artificiales de manera intuitiva. Link: https://keras.io/getting_started/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import Sequential, layers, Input

# Ejecutar para descargar y extraer los datasets.
!pip install -q gdown
!gdown --fuzzy "https://drive.google.com/file/d/16AB3ixX9q2S7fHAbiE7id7zj1fCgPbo4/view" -O /content/datasets_laboratorio.zip
!unzip /content/datasets_laboratorio.zip -d /content/

## Clasificación usando Redes Neuronales Artificiales

Las **Redes Neuronales Artificiales (RNAs)** son un método de aprendizaje automático que en los últimos años ha ganado gran popularidad debido a su flexibilidad para resolver distintos tipos de problemas basándose en datos.

A nivel general, una RNA es un grafo dirigido donde los nodos se denominan neuronas artificiales y se encuentran conectados por arcos. **Cada neurona y arco en el grafo tendrá un peso, que se irá ajustando automáticamente** a medida que transcurre el entrenamiento de la red.

Una RNA está compuesta por:
*   Una **capa de entrada**, que tendrá una neurona por cada atributo de los datos.
*   **Capas ocultas** (internas), sin restricciones de forma.
*   Una **capa de salida**, que tendrá una neurona por cada categoría de clasificación.

Existen muchos tipos de arquitecturas de RNAs, algunos destinados a resolver problemas con tipos de datos particulares. En esta sección vamos desarrollar una RNA feed-forward, útil para datos tabulares, que nos permita predecir el género de una canción.

### Carga y separación de los datos

In [ ]:
# Cargar el dataset
df_music = pd.read_csv('/content/music.csv')

print("Cantidad de ejemplos: {}\nCantidad de columnas: {}".format(df_music.shape[0], df_music.shape[1]))
df_music.head(5)

In [ ]:
df_music.info()

In [ ]:
print(df_music['class'].value_counts())

In [ ]:
print(df_music['class'].value_counts(normalize=True) * 100)

In [ ]:
# Separar la columna target
Y = df_music['class'].values
X = df_music.drop(columns = ["class"])

In [ ]:
# Porcentajes para datos de validación y test.
test_size = 0.15
val_size = 0.10

# La función train_test_split separa los datos (X, Y) en dos porciones
X_train, X_test, y_train, y_test = train_test_split(X, Y,
                                                    test_size = test_size,
                                                    random_state = 10,
                                                    stratify = Y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train,
                                                  test_size = val_size,
                                                  random_state = 10,
                                                  stratify = y_train)

# Nos guardamos los nombres de los atributos para más adelante
feature_names = X_train.columns.values[:-1]

print("Datos de entrenamiento:\n\tCantidad de ejemplos (filas): {}\n\tCantidad de atributos (columnas): {}\n".format(X_train.shape[0], X_train.shape[1]))
print("Datos de validación:\n\tCantidad de ejemplos (filas): {}\n\tCantidad de atributos (columnas): {}\n".format(X_val.shape[0], X_val.shape[1]))
print("Datos de test:\n\tCantidad de ejemplos (filas): {}\n\tCantidad de atributos (columnas): {}".format(X_test.shape[0], X_test.shape[1]))

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
#Imprimimos los porcentajes de las clases de Y, y_train, y_test e y_val
for nombre, datos in [
    ("Y", Y),
    ("y_train", y_train),
    ("y_val", y_val),
    ("y_test", y_test)
]:
    print(f"Proporciones de las clases de {nombre}:")
    print(
      (pd.Series(Y).value_counts(normalize=True) * 100)
      .sort_index()
      .round(2)
      .to_string(header=False), "\n"
    )

### Modificación de las etiquetas

Antes de pasar a construir y entrenar una RNA, es necesario que representemos las etiquetas (y_train, y_val, y_test) con alguna representación numérica. Recordar que hasta el momento, las etiquetas son strings (HipHop, Instrumental, Rock, Pop).

En este caso, vamos a usar el formato one-hot para representar las etiquetas del dataset. Para ello, usaremos un objeto de clase OneHotEncoder provisto por la librería scikit-learn.

Como resultado, podrás ver el mapeo que define el encoder para representar cada una de las etiquetas posibles. Cada categoría se representa como un vector de 4 valores (0s y 1s). La salida de la RNA al hacer una predicción tendrá esta misma forma.

**Link**:
* `https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html`

In [ ]:
# Onehot encoder
enc = OneHotEncoder()

# El encoder define un mapeo para las etiquetas.
y_train_enc = enc.fit_transform(y_train.reshape(-1, 1)).toarray()
y_val_enc = enc.transform(y_val.reshape(-1, 1)).toarray() # Aquí usar transform.
y_test_enc = enc.transform(y_test.reshape(-1, 1)).toarray() # Aquí usar transform.

print("Etiquetas (one-hot encoding):\n{}".format(y_train_enc))
print()

print("Mapeo del encoder:")
pd.DataFrame({
    c : enc.transform([[c]]).toarray()[0]
    for c in list(enc.categories_[0])
})

### Desarrollo de RNA

Comencemos desarrollando una RNA muy simple.

**Links**:
* Sequential Model: `https://keras.io/api/models/sequential/#sequential-class`
* Optimizadores en Keras: `https://keras.io/api/optimizers/#optimizers`
* Métricas en Keras: `https://keras.io/api/metrics/`
* Método compile: `https://keras.io/api/models/model_training_apis/#compile-method`
* Método fit: `https://keras.io/api/models/model_training_apis/#fit-method`

In [ ]:
num_features = X_train.shape[1] # cantidad de atributos en X_train y X_val
num_outputs = y_train_enc.shape[1] # cantidad de categorías posibles (4 géneros musicales)

def define_model_dummy(num_features, num_outputs):
    model = Sequential()

    # Capa de entrada de la red
    model.add(Input(shape=(num_features, ))) # tamaño de la entrada a la RNA

    model.add(layers.Dense(1, activation = 'relu'))

    # Capa de salida de la red
    model.add(layers.Dense(num_outputs, activation = 'softmax'))

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.1),
                  loss = 'categorical_crossentropy',
                  metrics = ['accuracy'])
    return model

# Se crea el modelo, aún sin entrenar
model = define_model_dummy(num_features, num_outputs)

# Obtenemos una descripción de la estructura
model.summary()

In [ ]:
import networkx as nx

def dibujar_red_neuronal(model, class_names=None):
    num_features = model.input_shape[-1]
    num_outputs = model.layers[-1].units

    if class_names is None:
        class_names = [f"Clase {i+1}" for i in range(num_outputs)]

    G = nx.DiGraph()

    entradas = [f"x{i+1}" for i in range(num_features)]
    oculta = "h1"
    salidas = [f"y{i+1}" for i in range(num_outputs)]

    G.add_edges_from((x, oculta) for x in entradas)
    G.add_edges_from((oculta, y) for y in salidas)

    pos = {}
    for i, nodo in enumerate(entradas):
        pos[nodo] = (0, i - (num_features - 1) / 2)

    pos[oculta] = (1, 0)

    for i, nodo in enumerate(salidas):
        pos[nodo] = (2, i - (num_outputs - 1) / 2)

    etiquetas = {nodo: nodo for nodo in entradas}
    etiquetas[oculta] = "h1\nReLU"
    etiquetas.update({
        nodo: f"{nombre}\nSoftmax"
        for nodo, nombre in zip(salidas, class_names)
    })

    fig, ax = plt.subplots(figsize=(10, 6))

    nx.draw_networkx_edges(
        G, pos, ax=ax, arrows=True,
        arrowsize=18, edge_color="gray",
        node_size=2000
    )

    nx.draw_networkx_nodes(
        G, pos, nodelist=entradas,
        node_color="lightgray", node_size=1800, ax=ax
    )
    nx.draw_networkx_nodes(
        G, pos, nodelist=[oculta],
        node_color="cornflowerblue", node_size=2200, ax=ax
    )
    nx.draw_networkx_nodes(
        G, pos, nodelist=salidas,
        node_color="lightgreen", node_size=2200, ax=ax
    )

    nx.draw_networkx_labels(
        G, pos, labels=etiquetas, font_size=9, ax=ax
    )

    ax.set_title("Arquitectura de la RNA")
    ax.axis("off")
    plt.show()


# Mostrar el modelo como grafo
dibujar_red_neuronal(model, class_names=enc.categories_[0])


In [ ]:
# Parámetros de entrenamiento
eps = 3 # Epochs. Cantidad de veces que el modelo recorre el conjunto completo de entrenamiento.
bs = 32 # Batch size. Cantidad de ejemplos que el modelo procesa antes de actualizar el peso de sus parámetros.

# Entrenar la RNA definida.
history = model.fit(
    X_train, y_train_enc,
    batch_size = bs,
    epochs = eps,
    validation_data=(X_val, y_val_enc)
)

In [ ]:
def dibujar_red_con_pesos(model, class_names=None):
    from matplotlib.lines import Line2D

    num_features = model.input_shape[-1]
    num_outputs = model.layers[-1].units

    capa_oculta = model.layers[0]
    capa_salida = model.layers[1]

    # Obtener pesos y sesgos aprendidos
    pesos_entrada, sesgos_oculta = capa_oculta.get_weights()
    pesos_salida, sesgos_salida = capa_salida.get_weights()

    if class_names is None:
        class_names = [f"Clase {i+1}" for i in range(num_outputs)]

    # Nombres de las funciones de activación
    activacion_oculta = capa_oculta.activation.__name__
    activacion_salida = capa_salida.activation.__name__

    G = nx.DiGraph()

    entradas = [f"x{i+1}" for i in range(num_features)]
    ocultas = [f"h{i+1}" for i in range(capa_oculta.units)]
    salidas = [f"y{i+1}" for i in range(num_outputs)]

    G.add_nodes_from(entradas + ocultas + salidas)

    # Conexiones: entrada -> capa oculta
    for i, x in enumerate(entradas):
        for j, h in enumerate(ocultas):
            G.add_edge(x, h, weight=pesos_entrada[i, j])

    # Conexiones: capa oculta -> salida
    for i, h in enumerate(ocultas):
        for j, y in enumerate(salidas):
            G.add_edge(h, y, weight=pesos_salida[i, j])

    # Posiciones de los nodos
    pos = {}

    for i, nodo in enumerate(entradas):
        pos[nodo] = (0, i - (num_features - 1) / 2)

    for i, nodo in enumerate(ocultas):
        pos[nodo] = (
            1,
            i - (len(ocultas) - 1) / 2
        )

    for i, nodo in enumerate(salidas):
        pos[nodo] = (
            2,
            i - (num_outputs - 1) / 2
        )

    # Etiquetas de las neuronas
    etiquetas = {x: x for x in entradas}

    for i, h in enumerate(ocultas):
        etiquetas[h] = (
            f"{h}\n{activacion_oculta}"
            f"\nb = {sesgos_oculta[i]:.3f}"
        )

    for j, y in enumerate(salidas):
        etiquetas[y] = (
            f"{class_names[j]}\n{activacion_salida}"
            f"\nb = {sesgos_salida[j]:.3f}"
        )

    # Etiquetas y propiedades visuales de las aristas
    pesos = np.array([
        datos["weight"]
        for _, _, datos in G.edges(data=True)
    ])

    max_peso = np.max(np.abs(pesos)) if len(pesos) else 0

    etiquetas_pesos = {}
    colores_aristas = []
    grosores_aristas = []

    for u, v, datos in G.edges(data=True):
        w = datos["weight"]

        etiquetas_pesos[(u, v)] = f"{w:.3f}"

        # Verde: peso positivo; rojo: peso negativo
        colores_aristas.append(
            "forestgreen" if w >= 0 else "firebrick"
        )

        grosores_aristas.append(1.0)

    fig, ax = plt.subplots(figsize=(15, 8))

    # Dibujar conexiones
    nx.draw_networkx_edges(
        G, pos,
        ax=ax,
        arrows=True,
        arrowsize=18,
        edge_color=colores_aristas,
        width=grosores_aristas,
        node_size=2400,
        connectionstyle="arc3,rad=0.05"
    )

    # Dibujar neuronas de entrada
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=entradas,
        node_color="lightgray",
        node_size=1800,
        ax=ax
    )

    # Dibujar neuronas ocultas
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=ocultas,
        node_color="cornflowerblue",
        node_size=2800,
        ax=ax
    )

    # Dibujar neuronas de salida
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=salidas,
        node_color="lightgreen",
        node_size=3000,
        ax=ax
    )

    # Etiquetas de nodos
    nx.draw_networkx_labels(
        G, pos,
        labels=etiquetas,
        font_size=8,
        ax=ax
    )

    # Etiquetas de pesos
    nx.draw_networkx_edge_labels(
        G, pos,
        edge_labels=etiquetas_pesos,
        font_size=8,
        label_pos=0.5,
        rotate=False,
        ax=ax
    )

    # Leyenda
    leyenda = [
        Line2D(
            [0], [0],
            color="forestgreen", lw=2,
            label="Peso positivo"
        ),
        Line2D(
            [0], [0],
            color="firebrick", lw=2,
            label="Peso negativo"
        )
    ]

    ax.legend(handles=leyenda, loc="upper left")
    ax.set_title("RNA: arquitectura, pesos y sesgos aprendidos")
    ax.axis("off")

    plt.tight_layout()
    plt.show()

dibujar_red_con_pesos(
    model,
    class_names=enc.categories_[0]
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Error
ax1.plot(history.history['loss'], label='Entrenamiento')
ax1.plot(history.history['val_loss'], label='Validación')
ax1.set_title('Error')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Error')
ax1.legend()

# Accuracy
ax2.plot(history.history['accuracy'], label='Entrenamiento')
ax2.plot(history.history['val_accuracy'], label='Validación')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predicciones sobre el conjunto de validación
predictions = model.predict(X_test)

# Reporte de clasificación
print(classification_report(np.argmax(y_test_enc, axis = 1),
                            np.argmax(predictions, axis = 1),
                            target_names = enc.categories_[0]))

In [ ]:
predictions

In [ ]:
X_test_with_predictions = X_test.copy()
X_test_with_predictions['class'] = np.argmax(y_test_enc, axis = 1)
X_test_with_predictions['predictions'] = np.argmax(predictions, axis = 1)
X_test_with_predictions

### Re-escalado de los atributos

Una técnica de pre-procesamiento de los datos que puede resultar importante es la normalización o reescalado de los atributos del dataset. Para algunos métodos de aprendizaje (los que se basan en el cálculo de distancias o en descenso de gradiente, por ejemplo), las diferencias entre las magnitudes de los atributos pueden afectar negativamente el aprendizaje.

Vamos a usar un objeto de clase MinMaxScaler provisto por la librería scikit-learn para realizar la transformación de los datos.

Noten que el scaler recibe un intervalo (min, max), en este caso (-1, 1). Todo valor del dataset se transformará a un nuevo valor en ese intervalo. Además, el scaler se ajusta con los datos de entrenamiento y transforma tanto los datos de entrenamiento como los de verificación.

**Link**:
* `https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html#sklearn.preprocessing.MinMaxScaler`

In [ ]:
# MinMaxScaler
scaler = MinMaxScaler(feature_range = (-1, 1))

# El scaler se ajusta sobre los datos de entrenamiento X_train.
X_train_scl = scaler.fit_transform(X_train)
X_val_scl = scaler.transform(X_val)
X_test_scl = scaler.transform(X_test)

# Podemos ver el conjunto de datos luego de realizar la transformación.
pd.DataFrame(X_train_scl, columns = X_train.columns).head()

In [ ]:
fig, axs = plt.subplots(3,2, figsize  = (6, 8))

# Gráficas para el atributo danceability (antes de la transformación (izq) y después (der)).
axs[0, 0].hist(X_train['danceability'], bins = 50)
axs[0, 0].set_title("Danceability")

axs[0, 1].hist(X_train_scl[:, 0], bins = 50)
axs[0, 1].set_title("Danceability")

# Gráficas para el atributo loudness (antes de la transformación (izq) y después (der)).
axs[1, 0].hist(X_train['loudness'], bins = 50)
axs[1, 0].set_title("Loudness")

axs[1, 1].hist(X_train_scl[:, 3], bins = 50)
axs[1, 1].set_title("Loudness")

# Gráficas para el atributo tempo (antes de la transformación (izq) y después (der)).
axs[2, 0].hist(X_train['tempo'], bins = 50)
axs[2, 0].set_title("Tempo")

axs[2, 1].hist(X_train_scl[:, -1], bins = 50)
axs[2, 1].set_title("Tempo")

plt.tight_layout()
plt.show()

In [ ]:
num_features = X_train_scl.shape[1] # cantidad de atributos en X_train y X_val
num_outputs = y_train_enc.shape[1] # cantidad de categorías posibles (4 géneros musicales)

def define_model(num_features, num_outputs):
    model = Sequential()

    # Capa de entrada de la red
    model.add(Input(shape=(num_features, ))) # tamaño de la entrada a la RNA

    model.add(layers.Dense(10, activation = 'relu'))
    model.add(layers.Dense(10, activation = 'relu'))

    # Capa de salida de la red
    model.add(layers.Dense(num_outputs, activation = 'softmax'))

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss = 'categorical_crossentropy',
                  metrics = ['accuracy'])
    return model

# Se crea el modelo, aún sin entrenar
model = define_model(num_features, num_outputs)

eps = 10
bs = 32

# Entrenar la RNA definida.
history = model.fit(X_train_scl, y_train_enc, batch_size = bs, epochs = eps, validation_data=(X_val_scl, y_val_enc))

# Predicciones sobre el conjunto de validación
predictions = model.predict(X_test_scl)

# Reporte de clasificación
print(classification_report(np.argmax(y_test_enc, axis = 1),
                            np.argmax(predictions, axis = 1),
                            target_names = enc.categories_[0]))

In [ ]:
def dibujar_rna(model):
    # Obtener las capas Dense del modelo.
    # Se excluyen las capas de entrada explícitas.
    capas = [
        layer for layer in model.layers
        if isinstance(layer, __import__(
            "tensorflow"
        ).keras.layers.Dense)
    ]

    if not capas:
        raise ValueError("El modelo no contiene capas Dense.")

    if len(capas) != len(model.layers):
        raise ValueError(
            "Esta función requiere un modelo compuesto "
            "exclusivamente por capas Dense. No se puede "
            "representar fielmente esta arquitectura."
        )

    # Obtener matrices de pesos y sesgos.
    parametros = []
    for capa in capas:
        pesos = capa.get_weights()

        if not pesos:
            raise ValueError(
                f"La capa {capa.name} no tiene pesos inicializados."
            )

        kernel = pesos[0]
        sesgos = pesos[1] if len(pesos) > 1 else None

        parametros.append((kernel, sesgos))

    # Verificar que las capas estén conectadas secuencialmente.
    for i in range(1, len(parametros)):
        if parametros[i - 1][0].shape[1] != parametros[i][0].shape[0]:
            raise ValueError(
                "Las dimensiones de las capas no son compatibles "
                "con una conexión secuencial directa."
            )

    # Número de neuronas de cada capa.
    num_entradas = parametros[0][0].shape[0]
    cantidades = [num_entradas] + [
        kernel.shape[1] for kernel, _ in parametros
    ]

    num_capas = len(cantidades)

    # Construir el grafo dirigido.
    G = nx.DiGraph()
    pos = {}
    etiquetas = {}
    entradas = []
    ocultas = []
    salidas = []

    # Crear nodos y sus posiciones.
    for l, cantidad in enumerate(cantidades):
        nodos = []

        for i in range(cantidad):
            nodo = f"L{l}_N{i}"
            nodos.append(nodo)
            G.add_node(nodo)

            # Distribuir las neuronas verticalmente.
            pos[nodo] = (
                l,
                (cantidad - 1) / 2 - i
            )

            if l == 0:
                entradas.append(nodo)
                etiquetas[nodo] = f"x{i + 1}"

            else:
                kernel, sesgos = parametros[l - 1]
                capa = capas[l - 1]

                activacion = capa.activation.__name__

                if sesgos is None:
                    sesgo_texto = "b = —"
                else:
                    sesgo_texto = f"b = {sesgos[i]:.3f}"

                etiquetas[nodo] = (
                    f"N{i + 1}\n"
                    f"{activacion}\n"
                    f"{sesgo_texto}"
                )

                if l == num_capas - 1:
                    salidas.append(nodo)
                else:
                    ocultas.append(nodo)

    # Agregar todas las conexiones y sus pesos.
    for l, (kernel, _) in enumerate(parametros):
        for i in range(kernel.shape[0]):
            for j in range(kernel.shape[1]):
                origen = f"L{l}_N{i}"
                destino = f"L{l + 1}_N{j}"

                G.add_edge(
                    origen,
                    destino,
                    weight=float(kernel[i, j])
                )

    # Dibujar.
    fig, ax = plt.subplots(
        figsize=(max(10, 3 * num_capas), 8)
    )

    nx.draw_networkx_edges(
        G, pos,
        ax=ax,
        arrows=True,
        arrowsize=12,
        edge_color="gray",
        width=1.0,
        node_size=2600,
        connectionstyle="arc3,rad=0.03"
    )

    nx.draw_networkx_nodes(
        G, pos,
        nodelist=entradas,
        node_color="lightgray",
        node_size=2200,
        ax=ax
    )

    nx.draw_networkx_nodes(
        G, pos,
        nodelist=ocultas,
        node_color="cornflowerblue",
        node_size=3000,
        ax=ax
    )

    nx.draw_networkx_nodes(
        G, pos,
        nodelist=salidas,
        node_color="lightgreen",
        node_size=3000,
        ax=ax
    )

    nx.draw_networkx_labels(
        G, pos,
        labels=etiquetas,
        font_size=8,
        ax=ax
    )

    # Mostrar el valor de cada peso sobre su conexión.
    etiquetas_arcos = {
        (u, v): f"{datos['weight']:.3f}"
        for u, v, datos in G.edges(data=True)
    }

    nx.draw_networkx_edge_labels(
        G, pos,
        edge_labels=etiquetas_arcos,
        font_size=7,
        rotate=False,
        label_pos=0.5,
        ax=ax
    )

    # Identificar las capas.
    for l in range(num_capas):
        ax.text(
            l,
            max(cantidades) / 2 + 1,
            (
                "Entrada" if l == 0
                else "Salida" if l == num_capas - 1
                else f"Oculta {l}"
            ),
            ha="center",
            fontsize=10,
            fontweight="bold"
        )

    ax.set_title("RNA: pesos, funciones de activación y sesgos")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


# Ejecutar después de entrenar el modelo.
dibujar_rna(model)


In [ ]:
# Definir una RNA, entrenarla y evaluarla.
# <TO-DO>